In [2]:
# prompt: mount drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Install required libraries

In [3]:
!apt-get update -y
!apt-get install -y fluidsynth libfluidsynth-dev

!pip install pyfluidsynth
!pip install pretty_midi
!pip install onnx onnxruntime

from scipy.signal import resample
from scipy.io.wavfile  import write as wav_write
import pretty_midi
import fluidsynth
import os
import random
import librosa
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset, Sampler
from torch.nn.utils.rnn import pad_sequence
import torch.nn as nn
import torch.nn.functional as F
import torch.onnx
from torch import optim
import matplotlib.pyplot as plt
import librosa.display
import soundfile as sf
from IPython.display import display, Audio

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,575 kB]
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [3,518 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [5,290 kB]
Get:13 https://developer.download.nvidia.c

## Preparation of audio material

In [ ]:
def export_all_instruments(Target_Samplerate):
    target_samplerate = Target_Samplerate
    target_length = target_samplerate * 2
    dir_path = f"/content/drive/MyDrive/Colab_Notebooks/GANSound/trainingData"
    os.makedirs(dir_path, exist_ok=True)

    for program in range(1, 89):  # プログラム番号 1〜88

        # generating midi
        music = pretty_midi.PrettyMIDI()
        instrument = pretty_midi.Instrument(program=program)
        note = pretty_midi.Note(velocity=100, pitch=60, start=0.0, end=2.0)
        instrument.notes.append(note)
        music.instruments.append(instrument)

        # synthesize with SoundFont
        audio_data = music.fluidsynth(sf2_path="/content/drive/MyDrive/Colab_Notebooks/GANSound/Touhou.sf2")

        # resampling
        n_samples = int(len(audio_data) * target_samplerate / 44100)
        # print(len(audio_data))
        audio_data_resampled = resample(audio_data, n_samples)
        # print(len(audio_data_resampled))

        # cuting out only the required length
        if len(audio_data_resampled) < target_length:
            print(f"⚠️ Skipped {program} (too short: {len(audio_data_resampled)} samples)")
            continue

        clipped = audio_data_resampled[:target_length]

        # export
        filename = os.path.join(dir_path, f"60note_{program}.wav")
        if not os.path.exists(filename):
            wav_write(filename, target_samplerate, (clipped * 32767).astype(np.int16))
            print(f"✔️ {filename} Exported ({target_length} samples)")
        else:
            print(f"⏭️ {filename} already exists.")

Target_Samplerate = 22050
export_all_instruments(Target_Samplerate)

## PreProcess

In [ ]:
class PaddedBatchSampler(Sampler):
    #Sampler that supplements the last batch to the specified size
    def __init__(self, dataset_size, batch_size):
        self.dataset_size = dataset_size
        self.batch_size = batch_size

    def __iter__(self):
        #Create an index list
        indices = list(range(self.dataset_size))

        # Calculate the number of samples needed to complete the last batch.
        remainder = self.dataset_size % self.batch_size
        padding_size = 0 if remainder == 0 else self.batch_size - remainder

        # Fill in the gaps by randomly selecting from existing data.
        if padding_size > 0:
            padding_indices = random.choices(indices, k=padding_size)
            indices = indices + padding_indices

        # shuffle the index
        random.shuffle(indices)
        return iter(indices)

    def __len__(self):
        # Returns the number of samples divisible by the batch size.
        padding_size = 0 if self.dataset_size % self.batch_size == 0 else self.batch_size - (self.dataset_size % self.batch_size)
        return self.dataset_size + padding_size

class AudioPreprocessor:
    def __init__(self):
        self.sample_rate = Target_Samplerate
        self.segment_length = 2 * self.sample_rate  # Number of samples for 2 seconds
        self.min_length = int(1.5 * self.sample_rate)  # Number of samples for 1.5 seconds
        self.n_mels = 128  #Height of output of melspectrogram (number of dimensions)
        self.hop_length = 512

        # The number of noise "How diverse the output can be" affects the number of dimensions in the latent space.
        self.n_noise = 256 #This will be the input value for generater.
        self.epochs = 200
        self.n_fft = 2048
        self.min_db = -80.0
        self.max_db = 0.0

    def load_audio(self, file_name):
       #Load audio file
        file_path = os.path.join(self.audio_path, file_name)
        audio, _ = librosa.load(file_path, sr=self.sample_rate)
        return audio

    def adjust_segment(self, audio):
        #Divide the audio data into 2-second segments and zero-pad as necessary.
        segments = []
        for i in range(0, len(audio), self.segment_length):
            segment = audio[i:i + self.segment_length]
            if len(segment) == self.segment_length:
                segments.append(segment)
            elif len(segment) >= self.min_length:
                # Zero padding for values greater than 1.5 seconds
                padded_segment = np.pad(segment, (0, self.segment_length - len(segment)), mode='constant')
                segments.append(padded_segment)
        return segments

    def preprocess_audio(self, audio):
        #Converting audio data to melspectrograms
        mel_spec = librosa.feature.melspectrogram(
            y=audio, sr=self.sample_rate, n_mels=self.n_mels, hop_length=self.hop_length, n_fft=self.n_fft
        )
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
        mel_spec_db = np.clip(mel_spec_db, self.min_db, self.max_db)
        mel_spec_db = 2 * (mel_spec_db - self.min_db) / (self.max_db - self.min_db) - 1

        return mel_spec_db

    def process_all_files(self, audio_path):
        #Process all audio files in the directory and return a mel-spectrogram.
        self.audio_path = audio_path
        all_mel_specs = []
        for file_name in os.listdir(audio_path):
            if file_name.endswith('.wav'):
                print(f"Processing {file_name}...")
                audio = self.load_audio(file_name)
                segments = self.adjust_segment(audio)
                mel_specs = [self.preprocess_audio(segment) for segment in segments]
                all_mel_specs.extend(mel_specs)
        print("All files processed.")
        return all_mel_specs

    def collate_fn(self, batch):
        # batch is a list of tuples (mel_spec)
        mel_specs = [item[0] for item in batch]

        # Pad mel_specs to the maximum length in the batch
        mel_specs_padded = pad_sequence(mel_specs, batch_first=True)
        return (mel_specs_padded,)

    def create_dataloader(self, mel_specs, batch_size=16):
        #Create a data loader from melspectrograms (complete the last batch as well)
        mel_specs_array = np.array(mel_specs)
        mel_specs_tensor = torch.tensor(mel_specs_array, dtype=torch.float)
        dataset = torch.utils.data.TensorDataset(mel_specs_tensor)

        # Use a custom sampler to fill in the gaps.
        sampler = PaddedBatchSampler(len(dataset), batch_size)

        dataloader = torch.utils.data.DataLoader(
            dataset,
            batch_size=batch_size,
            sampler=sampler,  # If you specify a sampler, shuffle will be ignored.
            collate_fn=self.collate_fn
        )

        #Calculate how many batches there will be in the last batch based on the dataset size and batch size.
        num_samples = len(dataset)
        num_complete_batches = num_samples // batch_size
        remaining_samples = num_samples % batch_size

        print(f"dataset size: {num_samples}")
        print(f"total number of batches: {num_complete_batches}")
        if remaining_samples > 0:
            print(f"Number of samples in the last batch: {remaining_samples} (Complemented to {batch_size})")
        else:
            print("All batches are complete.")

        return dataloader

    def get_trainloder(self):
        self.audio_path = "/content/drive/MyDrive/Colab_Notebooks/GANSound/trainingData"

        mel_specs = self.process_all_files(self.audio_path)
        print(f"Number of processed melspectrograms: {len(mel_specs)}")

        # Create a data loader (with completion feature)
        dataloader = self.create_dataloader(mel_specs, batch_size=16)


        #This is a tuple (tensor) ←tuple with 1 element.
        for batch in dataloader:
            print(f"Batch shape 1: {batch[0].shape}")
            x = batch[0]
            total_elements = x.shape[1] * x.shape[2]  # 128 * 87 = 11136
            break


        self.time_frames = x.shape[2]
        print(f"Batch shape 2: {x.shape}")

        print(f"Total number of elements in a batch: {total_elements}")

        return dataloader, total_elements

preprocessor = AudioPreprocessor()
train_loader, n_in_out = preprocessor.get_trainloder()

## Generator


In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        '''
        DCGAN For input size 𝑊, kernel size 𝐾, padding p, and stride s, ((𝑊−1)×𝑠−𝑝×2+𝐾)
        Starting with input noise, gradually expand using the inverse convolution operation.
        Calculate the initial size and work backwards: 128/32=4, 87/32≈2.7→3
        Therefore, the starting size is 4x3.
        '''

        # Projection layer - Create small feature maps from noise vectors
        self.projection = nn.Linear(preprocessor.n_noise, 256 * 4 * 3)
        self.bn_proj = nn.BatchNorm1d(256 * 4 * 3)

        # Expansion using convolutional transposition layers
        # 4x3 → 8x6
        self.convt1 = nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1)
        self.bn1 = nn.BatchNorm2d(128)

        # 8x6 → 16x11 ((6 - 1)* 2 - 0 + 3) = 13
        self.convt2 = nn.ConvTranspose2d(128, 64, kernel_size=(4, 3), stride=2, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        # 16x11 → 32x22
        self.convt3 = nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1)
        self.bn3 = nn.BatchNorm2d(32)

        # 32x22 → 64x44
        self.convt4 = nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1)
        self.bn4 = nn.BatchNorm2d(16)

        # 64x44 → 128x87
        self.convt5 = nn.ConvTranspose2d(16, 1, kernel_size=(4, 3), stride=2, padding=1)

    def forward(self, x):

        # Flatten to two dimensions before putting into Linear
        x = x.view(-1, preprocessor.n_noise)  # [batch_size, n_noise]

        # Projection & Reshape
        x = self.projection(x)                # [batch_size, 256*4*3]
        x = self.bn_proj(x)
        x = F.relu(x)
        x = x.view(-1, 256, 4, 3)             # [batch_size, 256, 4, 3]

        # Stepwise upscaling using reverse convolution layers
        x = F.relu(self.bn1(self.convt1(x)))  # [batch_size, 128, 8, 6]
        # print(f"1: {x.shape}")
        x = F.relu(self.bn2(self.convt2(x)))  # [batch_size, 64, 16, 11]
        # print(f"2: {x.shape}")
        x = F.relu(self.bn3(self.convt3(x)))  # [batch_size, 32, 32, 22]
        # print(f"3: {x.shape}")
        x = F.relu(self.bn4(self.convt4(x)))  # [batch_size, 16, 64, 44]
        # print(f"4: {x.shape}")

        # Final layer - Output restricted to [-1, 1] with tanh activation function
        x = torch.tanh(self.convt5(x))        # [batch_size, 1, 128, 87]
        # print(f"5: {x.shape}")
        return x

generator = Generator()
generator.cuda()  # GPU support
print(generator)

## Discriminator

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()

        '''
        DCGAN ((𝑊+{𝑝×2}−𝐾)/𝑠+1)
        Feature extraction and downsampling using convolutional layers
        In the DCGAN paper, BatchNorm is recommended for Discriminator layers other than the first layer.
        '''
        # 128x87 → 64x43　 ((87 + 2 - 4) / 2 + 1)
        self.conv1 = nn.Conv2d(1, 16, kernel_size=4, stride=2, padding=1)
        # LeakyReLU only (do not apply BatchNorm to the first layer - recommended by DCGAN paper)

        # 64x43 → 32x21　torch.Size([16, 32, 32, 26])
        self.conv2 = nn.Conv2d(16, 32, kernel_size=4, stride=2, padding=1)
        self.bn2 = nn.BatchNorm2d(32)

        # 32x21 → 16x10
        self.conv3 = nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1)
        self.bn3 = nn.BatchNorm2d(64)

        # 16x10 → 8x5
        self.conv4 = nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1)
        self.bn4 = nn.BatchNorm2d(128)

        # 8x5 → 4x2
        self.conv5 = nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1)
        self.bn5 = nn.BatchNorm2d(256)

        # final determination layer - 4x2 → 1x1
        self.conv6 = nn.Conv2d(256, 1, kernel_size=(4, 2), stride=1, padding=0)

        # Dropout (optional - improves model stability)
        self.dropout = nn.Dropout2d(0.3)


    def forward(self, x):
        #Convert to one dimension.

        x = x.view(x.size(0), 1, preprocessor.n_mels, -1)  #(batch size, number of channels, height, width)
        # print(f"Input shape Discriminator: {x.shape}")

        # Downsampling through a convolutional layer
        x = F.leaky_relu(self.conv1(x), negative_slope=0.2)
        # print(f"After conv1: {x.shape}")

        x = F.leaky_relu(self.bn2(self.conv2(x)), negative_slope=0.2)
        # print(f"After conv2: {x.shape}")

        # Add dropout (optional)
        x = self.dropout(x)

        x = F.leaky_relu(self.bn3(self.conv3(x)), negative_slope=0.2)
        # print(f"After conv3: {x.shape}")

        x = F.leaky_relu(self.bn4(self.conv4(x)), negative_slope=0.2)
        # print(f"After conv4: {x.shape}")

        x = F.leaky_relu(self.bn5(self.conv5(x)), negative_slope=0.2)
        # print(f"After conv5: {x.shape}")

        # Final layer - Outputs probabilities between 0 and 1 using sigmoid activation
        x = torch.sigmoid(self.conv6(x))
        # print(f"After conv6: {x.shape}")

        # Transform into a shape with a batch size * 1
        x = x.view(-1, 1)
        # print(f"Final output: {x.shape}")
        return x

discriminator = Discriminator()
discriminator.cuda()  # GPU support
print(discriminator)

## Generate Noise


In [ ]:
import matplotlib.pyplot as plt
#Generate and display images
def generate_images(i):

    n_rows = 4
    n_cols = 4
    noise = torch.randn(n_rows * n_cols, preprocessor.n_noise).cuda()
    g_imgs = generator(noise)
    g_imgs = g_imgs/2 + 0.5  # Set to a range of 0-1
    g_imgs = g_imgs.cpu().detach().numpy()

    time_frames_spaced = preprocessor.time_frames + 2
    n_mels_spaced = preprocessor.n_mels + 2

    # Overall image
    matrix_image = np.zeros((time_frames_spaced*n_rows, n_mels_spaced*n_cols))

    #  Arrange the generated images side by side to create a single image.
    for r in range(n_rows):
        for c in range(n_cols):
          # Convert to time × frequency resolution
            g_img = g_imgs[r*n_cols + c].reshape(preprocessor.time_frames, preprocessor.n_mels)
            top = r*time_frames_spaced
            left = c*n_mels_spaced
            matrix_image[top : top+preprocessor.time_frames, left : left+preprocessor.n_mels] = g_img

    plt.figure(figsize=(8, 8))
    plt.imshow(matrix_image, cmap="Greys_r", vmin=0.0, vmax=1.0)
    plt.tick_params(labelbottom=False, labelleft=False, bottom=False, left=False)  # Erase axis labels and lines
    plt.show()


# Calculation of correct answers
def count_correct(y, t):
    correct = torch.sum((torch.where(y<0.5, 0, 1) ==  t).float())
    return correct.item()

## Training

In [ ]:
def trainModel():
  # Binary cross-entropy error function
  loss_func = nn.BCELoss()

  # This is a hyperparameter of the Adam optimizer that specifies the decay rate of the exponential moving average.
  optimizer_gen = optim.Adam(generator.parameters(), lr=0.01, betas=(0.5, 0.999))
  optimizer_disc = optim.Adam(discriminator.parameters(), lr=0.0001, betas=(0.5, 0.999))

  error_record_fake = []  # Fake image error log
  acc_record_fake = []  # Fake image accuracy record
  error_record_real = []  # Authentic image error log
  acc_record_real = []  # Accuracy record of authentic images

  # Training
  generator.train()
  discriminator.train()
  for i in range(preprocessor.epochs):
      loss_fake = 0  # 誤差
      correct_fake = 0  # 正解数
      loss_real = 0
      correct_real = 0
      n_total = 0  # Total number of data (used for accuracy calculation)
      for j, (x,) in enumerate(train_loader):  # Extract mini batch (x,)

          # print("x max:", x.max().item(), "x min:", x.min().item())
          n_total += x.size()[0]  # Cumulative batch size

          # Generate images from noise and train the discriminator
          noise = torch.randn(x.size()[0], preprocessor.n_noise).cuda()

          imgs_fake = generator(noise)  # Image generation
          t = torch.zeros(x.size()[0], 1).cuda()  # The correct answer is 0.

          # print("first")
          y = discriminator(imgs_fake)

          # print("y ", y.shape)
          # print("t ", t.shape)
          loss = loss_func(y, t)
          optimizer_disc.zero_grad()
          loss.backward()
          optimizer_disc.step()  # Update only discriminator parameters
          loss_fake += loss.item()
          correct_fake += count_correct(y, t)

          # Train the discriminator using real images.
          imgs_real= x.cuda()
          t = torch.ones(x.size()[0], 1).cuda()  # The correct answer is 1.

          # print("second")
          y = discriminator(imgs_real)
          loss = loss_func(y, t)
          optimizer_disc.zero_grad()
          loss.backward()
          optimizer_disc.step()  # Update only discriminator parameters
          loss_real += loss.item()
          correct_real += count_correct(y, t)

          # Train the generator
          noise = torch.randn(x.size()[0]*2, preprocessor.n_noise).cuda()  # Double the batch size
          imgs_fake = generator(noise)  # Image generation
          t = torch.full((x.size()[0]*2, 1), 0.9).cuda()  # It looks real, but it's a little vague.
          # print("third")
          y = discriminator(imgs_fake)
          loss = loss_func(y, t)
          optimizer_gen.zero_grad()
          loss.backward()
          optimizer_gen.step()  # Update only generator parameters

      loss_fake /= j+1  # error
      error_record_fake.append(loss_fake)
      acc_fake = correct_fake / n_total  # accuracy
      acc_record_fake.append(acc_fake)

      loss_real /= j+1  # error
      error_record_real.append(loss_real)
      acc_real = correct_real / n_total  # accuracy
      acc_record_real.append(acc_real)

      # Display errors, accuracy, and generated images at regular intervals.
      if i % preprocessor.interval == 0:
          print ("Epochs:", i)
          # Discriminator error and accuracy for generated images (fakes)
          print ("Error_fake:", loss_fake , "Acc_fake:", acc_fake)
          # Discriminator error and accuracy for real images
          print ("Error_real:", loss_real , "Acc_real:", acc_real)
          generate_images(i)

  # Save trained model
  torch.save(generator.state_dict(), "/content/drive/MyDrive/Colab_Notebooks/GANSound/model/generator.pth")
  torch.save(discriminator.state_dict(), "/content/drive/MyDrive/Colab_Notebooks/GANSound/model/discriminator.pth")

  # export as onnx model
  generator.eval()
  dummy_input = torch.randn(1, preprocessor.n_noise).cuda()
  torch.onnx.export(generator,
                    dummy_input,
                    "/content/drive/MyDrive/Colab_Notebooks/GANSound/model/generator.onnx",
                    export_params=True,
                    opset_version=10,
                    do_constant_folding=True, # Constant folding optimization
                    input_names = ['input'],
                    output_names = ['output'],
                    dynamic_axes={'input' : {0 : 'batch_size'}, 'output' : {0 : 'batch_size'}})
  print("The model has been saved.")


if __name__ == "__main__":
    trainModel()

## Inference

In [ ]:
def inference_gan():
    # Preparing the Generator Model
    generator = Generator()
    generator.load_state_dict(torch.load("/content/drive/MyDrive/Colab_Notebooks/GANSound/model/generator.pth", map_location="cpu"))
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    generator.to(device)
    generator.eval()

    # Hyperparameters (same as VAE)
    n_noise = preprocessor.n_noise
    n_mels = preprocessor.n_mels
    sample_rate = preprocessor.sample_rate
    hop_length = preprocessor.hop_length
    n_fft = preprocessor.n_fft
    min_db = preprocessor.min_db
    max_db = preprocessor.max_db
    time_frames = preprocessor.time_frames

    # Generating latent variables from noise and generating speech
    z = torch.randn(1, n_noise).to(device)
    # print("潜在変数 z:", z)

    # Generating a Mer Spectrogram with Generator
    with torch.no_grad():
        y = generator(z).cpu().numpy()

    # If the shape is [batch, channels, mel, time], use reshape.
    print("y.shape:", y.shape)
    image = y.reshape(n_mels, time_frames)
    print("image.shape:", image.shape)

    # Displaying the Mer Spectrogram
    plt.figure(figsize=(4, 4))
    librosa.display.specshow(image, sr=sample_rate, hop_length=hop_length, x_axis='time', y_axis='mel')
    plt.colorbar(format="%+2.0f dB")
    plt.title("Generated Mel-Spectrogram (GAN)")
    plt.show()

    # Converting a mel-spectrogram to audio (dB scale → power spectrum)
    mel_spec_db = image * (max_db - min_db) + min_db
    mel_spec = librosa.db_to_power(mel_spec_db)

    # Voice restoration by Griffin-Lim
    audio = librosa.feature.inverse.mel_to_audio(
        mel_spec,
        sr=sample_rate,
        n_fft=n_fft,
        hop_length=hop_length,
        n_iter=32
    )

    audio = audio / (np.max(np.abs(audio)) + 1e-8)

    # play sound
    audio_obj = Audio(audio, rate=sample_rate)
    display(audio_obj)


    sf.write("/content/drive/MyDrive/Colab_Notebooks/GANSound/outputAuido/generated_gan_audio.wav", audio, sample_rate)
    print("Audio has been saved.")

if __name__ == "__main__":
    inference_gan()
